In [ ]:
%%capture
import os, re

# Install Unsloth the "friend way" (works on Colab + avoids common torch/xformers issues)
if "COLAB_" not in "".join(os.environ.keys()):
    !pip -q install unsloth
else:
    import torch
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip -q install --no-deps unsloth unsloth-zoo bitsandbytes accelerate transformers timm datasets pillow jiwer evaluate huggingface_hub trl
    !pip -q install addict
    !pip -q install --no-deps {xformers}

In [ ]:
# =========================
# 1) Mount Drive
# =========================
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# =========================
# 2) CONFIG
# =========================
import os

# Dataset path
DATA_ROOT = "/content/drive/MyDrive/OCR/UIT_HWDB_line"

TRAIN_DIR = os.path.join(DATA_ROOT, "train_data")
TEST_DIR  = os.path.join(DATA_ROOT, "test_data")

# Save fine-tuned model + checkpoints
SAVE_DIR = "/content/drive/MyDrive/OCR/deepseek_ocr_finetuned"
CKPT_DIR = "/content/drive/MyDrive/OCR/checkpoints"

# DeepSeek-OCR base model directory
DEEPSEEK_OCR_CODE_DIR = "/content/deepseek_ocr"

# Training knobs (safe defaults)
MAX_SEQ_LEN = 2048     
EPOCHS = 1
LR = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8
SAVE_STEPS = 20
LOG_STEPS = 10


In [ ]:
# ==========================================
# 3) Download DeepSeek-OCR code repo locally
# ==========================================
from huggingface_hub import snapshot_download

snapshot_download("unsloth/DeepSeek-OCR", local_dir=DEEPSEEK_OCR_CODE_DIR)

# Add it to python path so we can import deepseek_ocr.*
import sys
if DEEPSEEK_OCR_CODE_DIR not in sys.path:
    sys.path.append(DEEPSEEK_OCR_CODE_DIR)

print("DeepSeek-OCR code ready at:", DEEPSEEK_OCR_CODE_DIR)


.gitattributes: 0.00B [00:00, ?B/s]

README-checkpoint.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

assets/fig1.png:   0%|          | 0.00/396k [00:00<?, ?B/s]

assets/show1.jpg:   0%|          | 0.00/117k [00:00<?, ?B/s]

... (3 outputs left out)

configuration_deepseek_v2.py: 0.00B [00:00, ?B/s]

modeling_deepseekv2.py: 0.00B [00:00, ?B/s]

processor_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

DeepSeek-OCR code ready at: /content/deepseek_ocr


In [ ]:
# ==========================================================
# 4) Index dataset
# Rule:
#   train_data/<id>/{1.jpg..n.jpg, label.json}
#   test_data/<id>/{1.jpg..n.jpg, label.json}
# ==========================================================
import json
from tqdm import tqdm

def list_numeric_dirs(parent):
    dirs = []
    for name in os.listdir(parent):
        p = os.path.join(parent, name)
        if os.path.isdir(p) and name.isdigit():
            dirs.append(p)
    dirs.sort(key=lambda x: int(os.path.basename(x)))
    return dirs

def index_split(split_dir):
    pairs = []
    for folder in tqdm(list_numeric_dirs(split_dir), desc=f"Indexing {os.path.basename(split_dir)}"):
        label_path = os.path.join(folder, "label.json")
        if not os.path.exists(label_path):
            raise FileNotFoundError(f"Missing label.json in: {folder}")

        with open(label_path, "r", encoding="utf-8") as f:
            labels = json.load(f)  # {"1.jpg": "...", "2.jpg": "...", ...}

        
        for img_name, text in labels.items():
            img_path = os.path.join(folder, img_name)
            if not os.path.exists(img_path):
                raise FileNotFoundError(f"Label exists but image missing: {img_path}")
            pairs.append({"image_path": img_path, "text": text})

    return pairs

train_pairs = index_split(TRAIN_DIR)
test_pairs  = index_split(TEST_DIR)

print("Train samples:", len(train_pairs))
print("Test samples :", len(test_pairs))
print("Example:", train_pairs[0])


Indexing test_data: 100%|██████████| 6/6 [00:03<00:00,  1.76it/s]

Train samples: 7028
Test samples : 201
Example: {'image_path': '/content/drive/MyDrive/OCR/UIT_HWDB_line/train_data/1/1.jpg', 'text': 'KHÁI QUÁT VỀ BIỂN ĐẢO VIỆT NAM'}


In [ ]:
# Sample test and train
import random

print("Original train samples:", len(train_pairs))

random.seed(42)  # reproducibility
train_pairs = random.sample(train_pairs, len(train_pairs) // 4)
test_pairs = random.sample(test_pairs, len(test_pairs) // 4)

print("Reduced train samples :", len(train_pairs))
print("Reduced test samples  :", len(test_pairs))
print("Example:", train_pairs[0])
print("Example:", test_pairs[0])


Original train samples: 7028
Reduced train samples : 1757
Reduced test samples  : 50
Example: {'image_path': '/content/drive/MyDrive/OCR/UIT_HWDB_line/train_data/185/36.jpg', 'text': 'việc thiện.'}
Example: {'image_path': '/content/drive/MyDrive/OCR/UIT_HWDB_line/test_data/253/16.jpg', 'text': 'những căn chòi tạm bợ sống qua ngày. Ngẫm sao mà thương thế! Đáng lẽ phải được'}


In [ ]:
# =========================
# 5) Lazy Dataset
# =========================
from torch.utils.data import Dataset

class LazyOCRDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # Only return path + label text.
        # The collator will open the image lazily.
        return self.pairs[idx]

train_ds = LazyOCRDataset(train_pairs)
test_ds  = LazyOCRDataset(test_pairs)

In [ ]:
# =========================
# 6) Load model (Unsloth)
# =========================
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"

import unsloth
from unsloth import FastVisionModel

from transformers import AutoModel

MODEL_NAME = "unsloth/DeepSeek-OCR"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_NAME,
    auto_model = AutoModel,
    load_in_4bit = True,
    trust_remote_code = True,
    use_gradient_checkpointing = "unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.6: Fast Deepseekocr patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Deepseekocr does not support SDPA - switching to fast eager.


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at unsloth/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

In [ ]:
# Attach LoRA
model = FastVisionModel.get_peft_model(
    model,
    r = 16,                 # LoRA rank
    lora_alpha = 32,
    lora_dropout = 0,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias = "none",
    use_rslora = False,
    loftq_config = None
)

Unsloth: Making `model.base_model.model.model` require gradients


In [ ]:
# =====================================================
# 7) Create datacollator (Unsloth original contract)
#   - returns: images as list of tuples (images_crop, images_ori)
#   - builds messages from your dataset format: {"image_path", "text"}
# =====================================================

import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr.modeling_deepseekocr import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

def build_user_prompt():
    return "Vietnamese OCR. Please read the text in the image."

@dataclass
class DeepSeekOCRDataCollator:
    tokenizer: Any
    model: Any
    image_size: int = 640
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 640,
        base_size: int = 1024,
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
            normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        # BOS token
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    
    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data (PIL Image or bytes dict) to PIL Image RGB"""
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image = Image.open(io.BytesIO(image_data['bytes']))
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            # Determine crop ratio based on image size
            if image.size[0] <= 640 and image.size[1] <= 640:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num=2, max_num=9,
                    image_size=self.image_size, use_thumbnail=False
                )

            # Global view with padding
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            # Local views
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(self.image_transform(crop_img).to(self.dtype))

            # Tokenized image placeholder sequence
            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base + [self.image_token_id]) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num) + [self.image_token_id]) * (
                    num_queries * height_crop_num)

        else:
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            if self.base_size <= 640:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color=tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries + [self.image_token_id]) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    # Process a single sample
    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
        # Collect images from messages
        images = []
        for message in messages:
            if "images" in message and message["images"]:
                for img_data in message["images"]:
                    if img_data is not None:
                        images.append(self.deserialize_image(img_data))

        if not images:
            raise ValueError("No images found in sample.")

        tokenized_str = []
        images_seq_mask = []
        images_list, images_crop_list, images_spatial_crop = [], [], []

        prompt_token_count = -1
        assistant_started = False
        image_idx = 0

        # Add BOS
        tokenized_str.append(self.bos_id)
        images_seq_mask.append(False)

        for message in messages:
            role = message["role"]
            content = message["content"]

            if role == "<|Assistant|>":
                if not assistant_started:
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True
                content = f"{content.strip()} {self.tokenizer.eos_token}"

            # Split by <image>
            text_splits = content.split('<image>')

            for i, text_sep in enumerate(text_splits):
                tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                # FIX: some versions return list -> keep as list for extend
                if isinstance(tokenized_sep, torch.Tensor):
                    tokenized_sep = tokenized_sep.tolist()

                tokenized_str.extend(tokenized_sep)
                images_seq_mask.extend([False] * len(tokenized_sep))

                # Followed by an image?
                if i < len(text_splits) - 1:
                    if image_idx >= len(images):
                        raise ValueError("Found <image> token but no corresponding image.")

                    image = images[image_idx]
                    img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                    images_list.extend(img_list)
                    images_crop_list.extend(crop_list)
                    images_spatial_crop.extend(spatial_crop)

                    tokenized_str.extend(tok_img)
                    images_seq_mask.extend([True] * len(tok_img))
                    image_idx += 1

        if image_idx != len(images):
            raise ValueError(f"Found {len(images)} images but used {image_idx} '<image>' tokens.")

        if not assistant_started:
            # No assistant message -> mask all
            prompt_token_count = len(tokenized_str)

        images_ori = torch.stack(images_list, dim=0)
        images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)

        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim=0)
        else:
            # keep shape consistent
            images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)

        return {
            "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_tensor,
            "prompt_token_count": prompt_token_count,
        }

    # BATCH COLLATE 
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_data = []
        user_prompt = build_user_prompt()
        
        for feature in features:
            try:
                img = Image.open(feature["image_path"]).convert("RGB")
                gt  = feature["text"]

                messages = [
                    {"role": "<|User|>", "content": f"<image>\n{user_prompt}", "images": [img]},
                    {"role": "<|Assistant|>", "content": gt},
                ]

                processed = self.process_single_sample(messages)
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        # Pad sequences
        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)

        # Labels
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        labels[images_seq_mask] = -100

        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        # images as list of tuples
        images_batch = [(item['images_crop'], item['images_ori']) for item in batch_data]

        # spatial crop (concatenate across batch)
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }

# Create collator
collator = DeepSeekOCRDataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=640,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=True,
)


In [ ]:
# =========================
# 8) Train 
# =========================
from transformers import Trainer, TrainingArguments
from unsloth import is_bf16_supported

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

trainer = Trainer(
    model=model,
    train_dataset=train_ds,
    data_collator=collator,
    args=TrainingArguments(
        output_dir=CKPT_DIR,              
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        logging_steps=LOG_STEPS,

        # Precision
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),

        # Optim/memory
        optim="adamw_8bit",
        remove_unused_columns=False,
        dataloader_num_workers=0,
        report_to="none",

        # Checkpointing (Drive-safe)
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
    )
)

trainer.train(resume_from_checkpoint=True)

# Save final LoRA adapter + tokenizer to Drive
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved final model to:", SAVE_DIR)
print("Saved checkpoints to:", CKPT_DIR)


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([3, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([8, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([3, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE: 

Step,Training Loss
60,0.674400
70,0.576500
80,0.566300
90,0.627800
100,0.608100
110,0.606400


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([4, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE: 

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([7, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([9, 100, 1280])
BASE: 

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Saved final model to: /content/drive/MyDrive/OCR/deepseek_ocr_finetuned
Saved checkpoints to: /content/drive/MyDrive/OCR/checkpoints


In [ ]:
# =====================================================
# 9) Evaluate CER (Baseline vs Fine-tuned) + SAVE RESULTS
# =====================================================


!pip -q install jiwer
import os, sys, json, contextlib
import torch
from tqdm import tqdm
from jiwer import cer

# ---- Config ----
PROMPT = "<image>\nVietnamese OCR. Please read the text in the image."
BASE_SIZE  = 1024
IMAGE_SIZE = 640
CROP_MODE  = True
MODEL_NAME = "unsloth/DeepSeek-OCR"

EVAL_DIR = os.path.join(SAVE_DIR, "eval")
os.makedirs(EVAL_DIR, exist_ok=True)

# Mute the annoying BASE/PATCHES prints 
@contextlib.contextmanager
def mute_stdout():
    old = sys.stdout
    sys.stdout = open(os.devnull, "w")
    try:
        yield
    finally:
        sys.stdout.close()
        sys.stdout = old

# Eval (runs inference + saves predictions + CER) 
def eval_and_save_cer(
    model,
    tokenizer,
    pairs,
    tag: str,
    out_dir: str,
    max_eval: int = None,
):
    if max_eval is not None:
        pairs = pairs[:max_eval]

    preds, gts = [], []
    records = []  # per-sample outputs

    pred_path = os.path.join(out_dir, f"predictions_{tag}.jsonl")
    cer_path  = os.path.join(out_dir, f"cer_{tag}.txt")

    print(f"\n===== CER EVAL: {tag} | N={len(pairs)} =====")
    model.eval()

    with torch.no_grad(), open(pred_path, "w", encoding="utf-8") as fjsonl:
        for ex in tqdm(pairs, desc=f"Running {tag}"):
            img_path = ex["image_path"]
            gt_text  = ex["text"]

            try:
                with mute_stdout():
                    pred_text = model.infer(
                        tokenizer,
                        prompt=PROMPT,
                        image_file=img_path,
                        output_path=out_dir, 
                        base_size=BASE_SIZE,
                        image_size=IMAGE_SIZE,
                        crop_mode=CROP_MODE,
                        save_results=False, 
                        test_compress=True,
                        eval_mode=True,
                    )
            except Exception as e:
                print(f"[{tag}] infer failed on {img_path}: {e}")
                pred_text = ""

            pred_text = "" if pred_text is None else str(pred_text)
            gt_text   = "" if gt_text   is None else str(gt_text)

            preds.append(pred_text)
            gts.append(gt_text)

            # Write JSONL line immediately
            rec = {
                "image_path": img_path,
                "ground_truth": gt_text,
                "prediction": pred_text,
            }
            fjsonl.write(json.dumps(rec, ensure_ascii=False) + "\n")

    score = cer(gts, preds)

    with open(cer_path, "w", encoding="utf-8") as f:
        f.write(f"CER ({tag}) = {score:.6f}\n")

    print(f"✅ {tag} CER = {score:.6f}")
    print(f"📁 Saved predictions: {pred_path}")
    print(f"📁 Saved CER score  : {cer_path}")

    return score, pred_path, cer_path

# =====================================================
# Load baseline (no LoRA) and evaluate
# =====================================================
import unsloth
from unsloth import FastVisionModel
from transformers import AutoModel

os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"

baseline_model, baseline_tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    auto_model=AutoModel,
    load_in_4bit=True,
    trust_remote_code=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(baseline_model)

baseline_cer, baseline_pred_file, baseline_cer_file = eval_and_save_cer(
    baseline_model,
    baseline_tokenizer,
    test_pairs,
    tag="baseline",
    out_dir=EVAL_DIR,
)

# Free VRAM
del baseline_model
torch.cuda.empty_cache()



# =====================================================
# Load finetuned (LoRA) and evaluate
# =====================================================
finetuned_model, finetuned_tokenizer = FastVisionModel.from_pretrained(
    model_name=SAVE_DIR,
    auto_model=AutoModel,
    load_in_4bit=True,
    trust_remote_code=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(finetuned_model)

finetuned_cer, finetuned_pred_file, finetuned_cer_file = eval_and_save_cer(
    finetuned_model,
    finetuned_tokenizer,
    test_pairs,
    tag="finetuned",
    out_dir=EVAL_DIR,
)

# =====================================================
# (F) Compare
# =====================================================
print("\n====================")
print(f"Baseline CER  : {baseline_cer:.6f}")
print(f"Finetuned CER : {finetuned_cer:.6f}")
print(f"Δ Improvement : {baseline_cer - finetuned_cer:.6f}  (positive = better)")
print("====================\n")

print("Files saved:")
print(" -", baseline_pred_file)
print(" -", baseline_cer_file)
print(" -", finetuned_pred_file)
print(" -", finetuned_cer_file)

A new version of the following files was downloaded from https://huggingface.co/unsloth/DeepSeek-OCR:
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/unsloth/DeepSeek-OCR:
- deepencoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/unsloth/DeepSeek-OCR:
- modeling_deepseekocr.py
- conversation.py
- deepencoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and 

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.6: Fast Deepseekocr patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Deepseekocr does not support SDPA - switching to fast eager.


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at unsloth/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (8 outputs left out)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at unsloth/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



===== CER EVAL: finetuned | N=50 =====


Running finetuned: 100%|██████████| 50/50 [06:33<00:00,  7.87s/it]

✅ finetuned CER = 0.179282
📁 Saved predictions: /content/drive/MyDrive/OCR/deepseek_ocr_finetuned/eval/predictions_finetuned.jsonl
📁 Saved CER score  : /content/drive/MyDrive/OCR/deepseek_ocr_finetuned/eval/cer_finetuned.txt

Baseline CER  : 0.343129
Finetuned CER : 0.179282
Δ Improvement : 0.163847  (positive = better)

Files saved:
 - /content/drive/MyDrive/OCR/deepseek_ocr_finetuned/eval/predictions_baseline.jsonl
 - /content/drive/MyDrive/OCR/deepseek_ocr_finetuned/eval/cer_baseline.txt
 - /content/drive/MyDrive/OCR/deepseek_ocr_finetuned/eval/predictions_finetuned.jsonl
 - /content/drive/MyDrive/OCR/deepseek_ocr_finetuned/eval/cer_finetuned.txt
